# Week 1 — Battery electrode voltage dataset
**Project:** Predicting the average voltage of insertion electrodes with machine learning (Materials Project data).

**This notebook:**
1. Installs the packages
2. Downloads every insertion electrode from the Materials Project (Li, Na, K, Mg, Ca, Zn, Al …)
3. Cleans the data
4. Explores it with plots
5. Saves a clean dataset for Week 2 (baseline model)

**Before you start:** get a free API key at https://next-gen.materialsproject.org/api (log in → *API Key*).
Run the cells from top to bottom (Shift + Enter).

In [ ]:
# 1. Install packages (takes ~1–2 minutes on Colab)
!pip -q install mp-api pymatgen pandas matplotlib seaborn

In [ ]:
# 2. Enter your Materials Project API key (it is hidden while you type and not saved in the notebook)
from getpass import getpass
API_KEY = getpass("Materials Project API key: ")

## 3. Download the data
Each entry is one *electrode*: a host framework (e.g. FePO4) plus a working ion (e.g. Li) inserted step by step.
We download the properties we need and the **host structure** (needed later for the graph neural network).

In [ ]:
from mp_api.client import MPRester
import pandas as pd

FIELDS = [
    "battery_formula", "working_ion", "framework_formula", "formula_charge", "formula_discharge",
    "average_voltage", "capacity_grav", "capacity_vol", "energy_grav", "energy_vol",
    "max_delta_volume", "max_voltage_step", "num_steps", "nelements", "elements",
    "stability_charge", "stability_discharge", "fracA_charge", "fracA_discharge",
    "id_charge", "id_discharge", "material_ids", "host_structure",
]

with MPRester(API_KEY) as mpr:
    # The route name changed between mp-api versions; this works with both.
    rester = getattr(getattr(mpr, "materials", None), "insertion_electrodes", None) or mpr.insertion_electrodes
    docs = rester.search(fields=FIELDS)

print(f"Downloaded {len(docs)} electrodes")

rows = []
for d in docs:
    r = {f: getattr(d, f, None) for f in FIELDS}
    r["working_ion"] = str(r["working_ion"])
    r["elements"] = [str(e) for e in (r["elements"] or [])]
    r["material_ids"] = [str(m) for m in (r["material_ids"] or [])]
    r["id_charge"], r["id_discharge"] = str(r["id_charge"]), str(r["id_discharge"])
    s = r.pop("host_structure")
    r["host_structure_json"] = s.to_json() if s is not None else None
    rows.append(r)

raw = pd.DataFrame(rows)
raw.to_csv("electrodes_raw.csv", index=False)
raw.head()

## 4. Clean the data
Rules (each one printed so you can see how many entries it removes):
- remove entries with a missing voltage, capacity or structure
- keep physically sensible voltages (0–6 V)
- keep electrodes whose **discharged** state is reasonably stable (energy above hull ≤ 0.1 eV/atom), so the model learns from realistic materials
- remove exact duplicates (same framework + same ion)

In [ ]:
df = raw.copy()
def step(mask, why):
    global df
    before = len(df); df = df[mask].copy(); print(f"{why:<55} {before:>6} → {len(df):>6}")

step(df["average_voltage"].notna() & df["capacity_grav"].notna() & df["host_structure_json"].notna(),
     "drop missing voltage / capacity / structure")
step(df["average_voltage"].between(0, 6), "keep 0 V ≤ voltage ≤ 6 V")
step(df["stability_discharge"].fillna(1) <= 0.1, "keep discharged state E_hull ≤ 0.1 eV/atom")
before = len(df)
df = df.sort_values("stability_discharge").drop_duplicates(["framework_formula", "working_ion"])
print(f"{'drop duplicate framework + ion':<55} {before:>6} → {len(df):>6}")

df = df.reset_index(drop=True)
print("\nElectrodes per working ion:")
print(df["working_ion"].value_counts())

## 5. Explore the data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
ions = df["working_ion"].value_counts().index.tolist()

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.countplot(data=df, x="working_ion", order=ions, ax=ax[0], color="#4C72B0")
ax[0].set(title="Number of electrodes per ion", xlabel="Working ion", ylabel="Count")
sns.boxplot(data=df, x="working_ion", y="average_voltage", order=ions, ax=ax[1], color="#DD8452")
ax[1].set(title="Average voltage per ion", xlabel="Working ion", ylabel="Average voltage (V)")
plt.tight_layout(); plt.savefig("fig_counts_voltage.png", dpi=200); plt.show()

In [ ]:
main_ions = [i for i in ["Li", "Na", "K", "Mg"] if i in ions]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for ion in main_ions:
    sns.kdeplot(df.loc[df.working_ion == ion, "average_voltage"], ax=ax[0], label=ion, fill=True, alpha=0.25)
ax[0].set(title="Voltage distribution", xlabel="Average voltage (V)"); ax[0].legend()
sns.scatterplot(data=df[df.working_ion.isin(main_ions)], x="capacity_grav", y="average_voltage",
                hue="working_ion", s=12, alpha=0.6, ax=ax[1])
ax[1].set(title="Voltage vs. gravimetric capacity", xlabel="Capacity (mAh/g)", ylabel="Average voltage (V)", xlim=(0, 1000))
plt.tight_layout(); plt.savefig("fig_distributions.png", dpi=200); plt.show()

In [ ]:
# Which elements appear most often in the host frameworks? (helps later when we split train/test)
from collections import Counter
cnt = Counter(e for els, ion in zip(df["elements"], df["working_ion"]) for e in els if e != ion)
top = pd.Series(cnt).sort_values(ascending=False).head(20)
top.plot.bar(figsize=(10, 3.5), color="#55A868", title="Most common elements in host frameworks")
plt.ylabel("Number of electrodes"); plt.tight_layout(); plt.savefig("fig_elements.png", dpi=200); plt.show()

print("Voltage statistics per ion (V):")
df.groupby("working_ion")["average_voltage"].describe()[["count", "mean", "std", "min", "max"]].round(2)

## 6. Save the clean dataset
`electrodes_clean.csv` is the input for Week 2. Download it (and the figures) from the Files panel on the left,
or save them to Google Drive with the optional cell below.

In [ ]:
df.to_csv("electrodes_clean.csv", index=False)
print(f"Saved {len(df)} electrodes to electrodes_clean.csv")

In [ ]:
# Optional: copy the results to Google Drive so they are kept after the Colab session ends
from google.colab import drive
import shutil, os
drive.mount("/content/drive")
dst = "/content/drive/MyDrive/electrode_voltage_ml"
os.makedirs(dst, exist_ok=True)
for f in ["electrodes_raw.csv", "electrodes_clean.csv", "fig_counts_voltage.png", "fig_distributions.png", "fig_elements.png"]:
    if os.path.exists(f): shutil.copy(f, dst)
print("Copied to", dst)

## What to look at before Week 2
1. **How many electrodes per ion?** Li will dominate. The Na/K/Mg counts decide how ambitious the transfer test (Week 4) can be.
2. **Do voltages differ by ion?** Multivalent ions (Mg, Ca, Zn, Al) and K usually sit at different voltages from Li.
3. **Outliers:** very high voltages often come from fluorides or oxides with high-valent metals — that is chemistry, not an error.

Send me the printed counts per ion and the statistics table, and we will build the baseline model in Week 2.